# Scenario Playground Step 9: Leakage and Data-Quality Audit

Objective: inspect distributions, outliers, missingness, regimes, correlations, and shock ranges; prove point-in-time and split safety; correct the one structural oil-volatility issue; regenerate the matrix; and re-run every check.

The final-test period is locked. Its target values are not used in distributions, correlations, outlier review, shock ranges, or decisions.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.leakage_audit import (
    build_leakage_audit_artifacts,
    persist_leakage_audit_artifacts,
)

artifacts = build_leakage_audit_artifacts()
output_paths = persist_leakage_audit_artifacts(artifacts)
summary = artifacts["summary"]
summary["audit_sequence"], output_paths

## Audit correction: negative CL=F settlement

The April 20, 2020 `CL=F` close is a real market event, but arithmetic percentage-return volatility is not a coherent comparable scale when a futures price crosses zero. The audit excludes that one feature globally, retains all rows and all other oil features, and applies no clipping or winsorization.

In [ ]:
display(artifacts["oil_volatility_review"])
print(summary["oil_volatility_review"])
assert summary["audit_sequence"]["initial_feature_count"] == 59
assert summary["audit_sequence"]["final_feature_count"] == 58
assert summary["audit_sequence"]["re_audit_passed"]

## Distributions, missingness, and regimes

IQR columns are review flags only; they do not transform data. Sparse shock columns with zero IQR are reviewed through their event flags and dedicated shock-range table. Missingness for final-test rows is checked structurally, without using held-out target magnitudes.

In [ ]:
display(artifacts["feature_distributions"].sort_values("iqr_outlier_count", ascending=False).head(15))
display(artifacts["target_distributions"].sort_values("iqr_outlier_count", ascending=False))
display(artifacts["regime_coverage"])
print("Missing values across every year/column check:", int(artifacts["missingness_by_year"]["missing_count"].sum()))

## Eight user-facing shock ranges

Ranges use pre-final-test features only. Structural zero rows remain part of the training contract, and release flags distinguish no event from an event whose measured shock is exactly zero.

In [ ]:
shock_ranges = artifacts["scenario_shock_ranges"]
display(shock_ranges)
assert len(shock_ranges) == 8
assert (shock_ranges["nonzero_count"] >= 20).all()

## Correlation review

High correlations are expected among broad equity states and targets. They motivate Ridge regularization but do not constitute leakage when every feature uses only pre-shock information.

In [ ]:
display(artifacts["feature_correlations"].sort_values("absolute_correlation", ascending=False).head(15))
display(artifacts["target_correlations"].sort_values("absolute_correlation", ascending=False).head(12))
display(artifacts["feature_target_correlations"].sort_values("absolute_correlation", ascending=False).head(12))

## Leakage checklist and training gate

Approval applies only to fitting on `development_train` rows and evaluating initial out-of-sample predictions on `validation` rows. Learned imputation, scaling, clipping, or any other preprocessing must be fit on development training data only. The 114 final-test rows remain locked.

In [ ]:
checklist = artifacts["leakage_checklist"]
print(checklist.to_string(index=False))
assert checklist["passed"].all()
assert summary["training_approval"] == "APPROVED_FOR_STEP_10_RIDGE_TRAINING"
print("STEP_9_COMPLETE_AND_MATRIX_APPROVED_FOR_RIDGE")